In [0]:
from pyspark.sql import DataFrame, functions as F
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.isotonic import IsotonicRegression

SILVER_TABLE = "stryktipset.silver.matches"
GOLD_TABLE = "stryktipset.gold.fact_match"  # built by 04_gold_fact_match

OUTCOMES = ("1", "X", "2")

In [0]:
def load_matches(table: str = SILVER_TABLE) -> DataFrame:
    """Load the match dataset written by 02_silver_transform."""
    return spark.table(table)

In [0]:
def to_long_format(df: DataFrame) -> DataFrame:
    """One row per (match, outcome): predicted_prob is the streck value,
    outcome is '1'/'X'/'2', actual_occurred is True when that outcome is
    what actually happened."""
    return (
        df.unpivot(
            ids=["draw_number", "event_number", "match_id", "result"],
            values=["streck_1", "streck_x", "streck_2"],
            variableColumnName="streck_col",
            valueColumnName="predicted_prob",
        )
        .withColumn(
            "outcome",
            F.upper(F.regexp_replace("streck_col", "^streck_", "")),
        )
        .withColumn("actual_occurred", F.col("outcome") == F.col("result"))
        .orderBy("draw_number", "event_number")
    )

In [0]:
def fit_calibration_curve(df_long: DataFrame, outcome: str) -> IsotonicRegression:
    """Fit one outcome's calibration curve. Done separately per outcome
    ('1'/'X'/'2') because the crowd's over/under-confidence isn't the same
    for home/away wins as it is for draws."""
    subset = (
        df_long.filter(F.col("outcome") == outcome)
        .select("predicted_prob", "actual_occurred")
        .toPandas()
    )

    x = subset["predicted_prob"].to_numpy()
    y = subset["actual_occurred"].to_numpy().astype(float)

    model = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
    model.fit(x, y)

    return model

In [0]:
def fit_all_calibration_curves(df_long: DataFrame) -> dict[str, IsotonicRegression]:
    """Returns {'1': model, 'X': model, '2': model}."""
    return {outcome: fit_calibration_curve(df_long, outcome) for outcome in OUTCOMES}

In [0]:
def calibrate_matches(df: DataFrame, models: dict[str, IsotonicRegression]) -> DataFrame:
    """Adds calibrated_1/calibrated_x/calibrated_2, rescaled to sum to 1
    per match."""
    sc = df.sparkContext
    broadcasts = {outcome: sc.broadcast(model) for outcome, model in models.items()}

    def make_predict_udf(outcome: str):
        bc = broadcasts[outcome]

        @F.pandas_udf("double")
        def _predict(raw: pd.Series) -> pd.Series:
            return pd.Series(bc.value.predict(raw.to_numpy()))

        return _predict

    raw_calibrated = (
        df.withColumn("calibrated_1", make_predict_udf("1")(F.col("streck_1")))
        .withColumn("calibrated_x", make_predict_udf("X")(F.col("streck_x")))
        .withColumn("calibrated_2", make_predict_udf("2")(F.col("streck_2")))
    )

    row_sum = F.col("calibrated_1") + F.col("calibrated_x") + F.col("calibrated_2")

    return (
        raw_calibrated
        .withColumn("calibrated_1", F.col("calibrated_1") / row_sum)
        .withColumn("calibrated_x", F.col("calibrated_x") / row_sum)
        .withColumn("calibrated_2", F.col("calibrated_2") / row_sum)
    )

In [0]:
def evaluate_calibration(df_calibrated: DataFrame) -> dict[str, float]:
    """Prints raw vs. calibrated Brier score per outcome + pooled, and
    returns the same numbers so main() can log them to MLflow."""
    column_pairs = {
        "1": ("streck_1", "calibrated_1"),
        "X": ("streck_x", "calibrated_x"),
        "2": ("streck_2", "calibrated_2"),
    }

    metrics: dict[str, float] = {}

    print(f"{'outcome':<8}{'raw brier':>12}{'calibrated brier':>20}")
    for outcome, (raw_col, calibrated_col) in column_pairs.items():
        actual = (F.col("result") == outcome).cast("double")
        row = df_calibrated.select(
            F.avg(F.pow(F.col(raw_col) - actual, 2)).alias("raw_brier"),
            F.avg(F.pow(F.col(calibrated_col) - actual, 2)).alias("calibrated_brier"),
        ).first()

        print(f"{outcome:<8}{row['raw_brier']:>12.4f}{row['calibrated_brier']:>20.4f}")
        metrics[f"brier_raw_{outcome}"] = row["raw_brier"]
        metrics[f"brier_calibrated_{outcome}"] = row["calibrated_brier"]

    pooled_raw = sum(metrics[f"brier_raw_{o}"] for o in OUTCOMES) / len(OUTCOMES)
    pooled_calibrated = sum(metrics[f"brier_calibrated_{o}"] for o in OUTCOMES) / len(OUTCOMES)
    print(f"{'pooled':<8}{pooled_raw:>12.4f}{pooled_calibrated:>20.4f}")
    metrics["brier_raw_pooled"] = pooled_raw
    metrics["brier_calibrated_pooled"] = pooled_calibrated

    return metrics

In [0]:
def merge_into_fact_match(df_calibrated: DataFrame, table: str = GOLD_TABLE) -> None:
    """Updates calibrated_1/x/2 on existing fact_match rows, matched by
    match_id. Requires 04_gold_fact_match to have already run."""
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

    source_view = "calibration_source"
    df_calibrated.select(
        "match_id", "calibrated_1", "calibrated_x", "calibrated_2"
    ).createOrReplaceTempView(source_view)

    spark.sql(f"""
        MERGE INTO {table} AS target
        USING {source_view} AS source
        ON target.match_id = source.match_id
        WHEN MATCHED THEN UPDATE SET
            target.calibrated_1 = source.calibrated_1,
            target.calibrated_x = source.calibrated_x,
            target.calibrated_2 = source.calibrated_2
    """)

In [0]:
def main():
    df = load_matches()
    df_long = to_long_format(df)

    with mlflow.start_run(run_name="calibration_v1"):
        models = fit_all_calibration_curves(df_long)
        df_calibrated = calibrate_matches(df, models).cache()

        mlflow.log_param("training_rows", df.count())
        metrics = evaluate_calibration(df_calibrated)
        mlflow.log_metrics(metrics)
        for outcome, model in models.items():
            mlflow.sklearn.log_model(model, f"isotonic_{outcome}")

        merge_into_fact_match(df_calibrated)
        print(f"Merged calibrated probabilities into {GOLD_TABLE}")

        df_calibrated.unpersist()

main()